# 7 · Widgets

A widget is the control a person edits an input with. Every input names its own; no type has a default.

In [ ]:
import json
from typing import Annotated

from conductor import CompiledGraph, Graph, Input, NodeRegistry, Param, Result, Series, Table, node
from conductor.widgets import (
    AnyWidget,
    Choice,
    CodeEditor,
    DatePicker,
    Dropdown,
    EntityDropdown,
    FileUpload,
    IfElseBuilder,
    ListWidget,
    NumberWidget,
    OperatorChoice,
    Range,
    SchemaBuilder,
    Switch,
    TableInput,
    Tags,
    TemplateTextarea,
    Textarea,
    TextWidget,
)
from pydantic import TypeAdapter

## Widgets with constraints

The widget goes on the `Param`, beside the `title`, `description` and `connectable` that belong to the field.

In [ ]:
PRIORITIES = (
    Choice(id="low", title="Low"),
    Choice(id="normal", title="Normal"),
    Choice(id="high", title="High"),
)


@node("send-email", title="Send Email", description="Send an email", category="mail")
def send_email(
    to: Annotated[str, Param(title="Recipient", widget=TextWidget(pattern=r"^[^@]+@[^@]+$"))],
    subject: Annotated[str, Param(title="Subject", widget=TextWidget(max_length=200))],
    body: Annotated[str, Param(title="Body", widget=Textarea(rows=8, min_length=10))],
    priority: Annotated[str, Param(title="Priority", widget=Dropdown(choices=PRIORITIES))] = "normal",
    retries: Annotated[int, Param(title="Retry attempts", widget=Range(min_val=0, max_val=5, step=1))] = 1,
) -> Annotated[str, Result(title="Status")]:
    return f"sent: {to=}, {subject=}, {priority=}, {retries=}"


for inp in send_email.versions[1].interface.inputs:
    print(f"  {inp.name:<10} kind={inp.widget.kind:<10} title={inp.title!r}  required={inp.required}")

## Every widget

`Param(connectable=False)` closes an input to edges, and it may then be any type pydantic can validate. A `Series[X]` input takes `ListWidget`, `Tags`, or no widget when an edge fills it.

In [ ]:
OPERATORS = (
    OperatorChoice(id="is-empty", title="is empty", category="text", arity=1),
    OperatorChoice(id="contains", title="contains", category="text", arity=2),
)


@node("widget-showcase", title="Showcase", description="Every widget", category="demo")
def showcase(
    # str & code
    heading: Annotated[str, Param(title="Title", widget=TextWidget(max_length=100))],
    notes: Annotated[str, Param(title="Notes", widget=Textarea(rows=6))],
    template: Annotated[str, Param(title="Template", widget=TemplateTextarea(rows=3))],
    code_snippet: Annotated[str, Param(title="Code", widget=CodeEditor(language="python"))],
    # Choice
    style: Annotated[str, Param(title="Style", widget=Dropdown(choices=(Choice(id="a", title="A"), Choice(id="b", title="B"))))],
    user: Annotated[str, Param(title="User", widget=EntityDropdown(entity_kind="user"))],
    # Numeric
    count: Annotated[int, Param(title="Count", widget=NumberWidget(integer_only=True, min_val=0))],
    volume: Annotated[float, Param(title="Volume", widget=Range(min_val=0, max_val=1, step=0.01))],
    # Boolean
    enabled: Annotated[bool, Param(title="Enabled", widget=Switch())],
    # Date & file
    starts: Annotated[str, Param(title="Starts", widget=DatePicker(min_date="2024-01-01", seed="today"))],
    upload: Annotated[str, Param(title="Upload", widget=FileUpload(accept=(".pdf", ".csv"), max_size_mb=10))],
    # Collections
    items: Annotated[Series[str], Param(title="Items", widget=ListWidget(min_items=1, max_items=10))],
    labels: Annotated[Series[str], Param(title="Labels", widget=Tags())],
    sources: Annotated[Series[str], Param(title="Sources")],
    # Structured
    table: Annotated[Table, Param(title="Table", widget=TableInput(min_rows=1, min_columns=2))],
    config: Annotated[dict, Param(title="Config", connectable=False, widget=SchemaBuilder(schema={"type": "object"}))],
    routing: Annotated[dict, Param(title="Routing", connectable=False, widget=IfElseBuilder(operators=OPERATORS))],
) -> Annotated[str, Result(title="Summary")]:
    return "all good"


inputs = showcase.versions[1].interface.inputs
print(f"{len(inputs)} inputs, {len({inp.widget.kind for inp in inputs if inp.widget})} distinct widget kinds:\n")
for inp in inputs:
    closed = "" if inp.connectable else "  (closed to edges)"
    kind = inp.widget.kind if inp.widget else "none"
    print(f"  {inp.name:<13} {inp.dtype.id:<8} widget={kind}{closed}")

## The dump

`AnyWidget` is the union of every widget, discriminated by `kind`.

In [ ]:
w = Range(min_val=0, max_val=100, step=5)
print(json.dumps(TypeAdapter(AnyWidget).dump_python(w, mode="json"), indent=2))

In [ ]:
# The same widget as its Input carries it: title and description sit beside it.
volume = next(inp for inp in inputs if inp.name == "volume")
print(json.dumps(TypeAdapter(Input).dump_python(volume, mode="json"), indent=2))

## Running a graph

A typed-in value is checked against its input's type.

In [ ]:
@node("label", title="Label", description="Glue", category="demo")
def label(
    n: Annotated[int, Param(title="N", widget=NumberWidget(integer_only=True))],
    name: Annotated[str, Param(title="Label", widget=TextWidget())],
) -> Annotated[str, Result(title="Out")]:
    return f"{name}={n}"


compiled = CompiledGraph.from_graph(Graph(nodes=[label.graph_node("n", n=42, name="answer")]), NodeRegistry([label]))
compiled.results(await compiled.run())["n"]["result"]